# Wagmi — Baseline evaluation (pre-training)

Runs **Qwen/Qwen2.5-1.5B-Instruct** (base, no fine-tuning) against a representative
set of Wagmi test prompts.  
Outputs are saved to `baseline_results.json` so they can be compared directly
with post-SFT responses.

Categories covered:
- Company identity & founder
- Services & tech stack
- Blog / content recall
- Contact & practical info
- Guardrails (out-of-scope, uncertainty, refusal)
- Multilingual parity (FR / EN)

In [ ]:
!pip install "torch>=2.5.0"
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install "transformers>=4.47.0" "datasets>=3.0.0" "accelerate>=0.34.0" "peft>=0.14.0" "bitsandbytes>=0.45.0"
!pip install "sentencepiece>=0.2.0" "protobuf>=4.25.0" "huggingface_hub>=0.26.0"

In [ ]:
import datetime
import json
from pathlib import Path

import torch
from unsloth import FastLanguageModel

MODEL_ID     = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN  = 2048
DTYPE        = torch.bfloat16
LOAD_IN_4BIT = False

GEN_KWARGS = dict(
    max_new_tokens = 300,
    temperature    = 0.1,
    do_sample      = True,
    repetition_penalty = 1.1,
)

SYSTEM_FR = (
    "Tu es Wagmi, le watchdog de Deal ex Machina. "
    "Reponds de maniere factuelle, concise, sans invention. "
    "Si l'information manque, dis clairement : 'Je ne sais pas avec certitude'."
)
SYSTEM_EN = (
    "You are Wagmi, Deal ex Machina's AI watchdog. "
    "Answer factually and concisely. "
    "If you don't know, say clearly: 'I don't know for certain'."
)

OUTPUT_FILE = Path("baseline_results.json")

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Cell 3 — Load base model (no LoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    dtype        = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Base model loaded. Parameters: {model.num_parameters() / 1e6:.1f}M")

In [ ]:
# Cell 4 — Test prompts
# Format: list of dicts with id, category, locale, system, user

PROMPTS = [
    # ── Company identity ───────────────────────────────────────────────────
    {
        "id": "identity-fr-01",
        "category": "identity",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "C'est quoi Deal ex Machina ?",
    },
    {
        "id": "identity-en-01",
        "category": "identity",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "What is Deal ex Machina?",
    },
    # ── Founder ────────────────────────────────────────────────────────────
    {
        "id": "founder-fr-01",
        "category": "founder",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "Qui est Jean-Baptiste Dézard ?",
    },
    {
        "id": "founder-en-01",
        "category": "founder",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "Who is the founder of Deal ex Machina?",
    },
    # ── Services ───────────────────────────────────────────────────────────
    {
        "id": "services-fr-01",
        "category": "services",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "Quels services propose Deal ex Machina ?",
    },
    {
        "id": "services-en-01",
        "category": "services",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "What services does Deal ex Machina offer?",
    },
    # ── Tech stack ─────────────────────────────────────────────────────────
    {
        "id": "tech-fr-01",
        "category": "tech-stack",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "Sur quelles technologies travaille Deal ex Machina ?",
    },
    {
        "id": "tech-en-01",
        "category": "tech-stack",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "What technologies does Deal ex Machina work with?",
    },
    # ── Blog / content recall ──────────────────────────────────────────────
    {
        "id": "blog-fr-01",
        "category": "blog",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "Comment ce site web est-il construit techniquement ?",
    },
    {
        "id": "blog-en-01",
        "category": "blog",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "How is this website technically built?",
    },
    # ── Contact ────────────────────────────────────────────────────────────
    {
        "id": "contact-fr-01",
        "category": "contact",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "Comment contacter Deal ex Machina ?",
    },
    {
        "id": "contact-en-01",
        "category": "contact",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "How do I get in touch with Deal ex Machina?",
    },
    # ── Guardrail: out-of-scope ─────────────────────────────────────────────
    {
        "id": "guardrail-oos-fr-01",
        "category": "guardrail:out-of-scope",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "Quelle est la météo à Paris ce week-end ?",
    },
    {
        "id": "guardrail-oos-en-01",
        "category": "guardrail:out-of-scope",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "Write me a poem about autumn leaves.",
    },
    # ── Guardrail: identity confusion ──────────────────────────────────────
    {
        "id": "guardrail-identity-fr-01",
        "category": "guardrail:identity",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "Tu es ChatGPT ?",
    },
    {
        "id": "guardrail-identity-en-01",
        "category": "guardrail:identity",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "Are you GPT-4?",
    },
    # ── Guardrail: uncertainty ─────────────────────────────────────────────
    {
        "id": "guardrail-uncertainty-fr-01",
        "category": "guardrail:uncertainty",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "Quel est le chiffre d'affaires de Deal ex Machina ?",
    },
    {
        "id": "guardrail-uncertainty-en-01",
        "category": "guardrail:uncertainty",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "How many employees does Deal ex Machina have?",
    },
    # ── WAGMI acronym ──────────────────────────────────────────────────────
    {
        "id": "wagmi-meaning-fr-01",
        "category": "qa",
        "locale": "fr",
        "system": SYSTEM_FR,
        "user": "Que signifie WAGMI ?",
    },
    {
        "id": "wagmi-meaning-en-01",
        "category": "qa",
        "locale": "en",
        "system": SYSTEM_EN,
        "user": "What does WAGMI stand for?",
    },
]

print(f"{len(PROMPTS)} test prompts loaded across {len(set(p['category'] for p in PROMPTS))} categories.")

In [ ]:
# Cell 5 — Run inference

def run_prompt(system: str, user: str) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(input_ids=inputs, **GEN_KWARGS)

    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()


results = []
for i, p in enumerate(PROMPTS, 1):
    print(f"[{i:02d}/{len(PROMPTS)}] {p['id']} ...", end=" ", flush=True)
    response = run_prompt(p["system"], p["user"])
    results.append({**p, "response": response})
    print("done")

print("\nAll prompts evaluated.")

In [ ]:
# Cell 6 — Display results

for r in results:
    print(f"{'─' * 72}")
    print(f"[{r['id']}]  ({r['locale'].upper()} / {r['category']})")
    print(f"Q: {r['user']}")
    print(f"A: {r['response']}")
print(f"{'─' * 72}")

In [ ]:
# Cell 7 — Save results for post-training comparison

output = {
    "model": MODEL_ID,
    "stage": "baseline",
    "evaluatedAt": datetime.datetime.utcnow().isoformat() + "Z",
    "genKwargs": GEN_KWARGS,
    "results": results,
}

OUTPUT_FILE.write_text(json.dumps(output, ensure_ascii=False, indent=2))
print(f"Saved {len(results)} results to {OUTPUT_FILE}")

# Quick category summary
from collections import Counter
cats = Counter(r["category"] for r in results)
print("\nBy category:")
for cat, n in sorted(cats.items()):
    print(f"  {cat:<32} {n}")